# w9_ps_swin.ipynb — 异步 PS × swin(@4096):为 swin 提速而生

User (2026-07-21): 异步的设计初衷就是解决 swin 训练慢。worker = **swin 步**
(now 锚 + 双半重叠窗微通道 + 立即反传;每 worker 环指针按 ps_id 错开,
K 个 worker 联合扫环 K 倍快);master 照旧 DC 补偿,`--ps-avg A` 可开
同批平均(每份先补偿到当前权重再平均,一组一步,lr×√A;epoch 按**消耗
推送数**对齐,协议数据量不变)。swin@4096 单 worker ~45G → 80G 卡一
worker,K=卡数;预期墙钟 ≈ 从零 swin / K。对照: 固定分割 swin 从零
(.691 @ep1150)。首跑 A=1(逐推送,与 sync 严格同更新数);A=K 平均模
式二跑。AUTO-STOPS。

**v2 分池(2026-07-21,用户"swin 的窗口是切分的")**: 目录随机切 K 个不相
交样本池,**每个 swin worker 的窗口环=自家池**——采样、负样本窗、扫环
(真规模下含数据装载)全部池内完成;分池为发布版本号纯函数,每 R=50ep
全体锁步轮换,跨池对由轮换重耦合。产物名 `..._psdc5[aA]sh50_fp`。

**v3(2026-07-21,用户完整设计)**: ①采样与窗口环都在自家池;②K=5 显式,
**cover=0.3 两段采样**(先随机指派家池,再向他池借样至 borrowed/|pool|=cover
——重叠即跨池胶水,取代轮换);③worker 切片自家池锚张量后**释放全量画廊**
(nonlocal 置 None,45G→池份额,单卡多 worker 成立;分配写 ps_pool_{id}.npz,
池外锚物理不可达);④**epoch 粒度推送**(冻结权重上累积 16 步,推均值);
⑤**master 硬屏障**: 集齐每个 worker 的新鲜梯度→逐份 DC 补偿到当前版本→
平均→单步(lr×√K),一轮=一 epoch,每轮必发布。产物 `..._psdc5shc30be_fp`。
预期陈旧度 ~1-3 时间片(慢 worker 由泰勒补偿吸收)。


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"

ARM = "wcle_swin168step84loop2i2ce_icetf"
CAP, EPOCHS, DC_LAMBDA, PS_AVG = 4096, 2000, 0.5, 1
SHARD, RESHUFFLE = True, 50
NWORKERS, COVER = 5, 0.3       # v3: K workers, overlap factor (static pools)
BARRIER, EPOCH_PUSH = True, True   # collect a fresh epoch-grad from EVERY worker per update
PS_DIR = "/dev/shm/w9_ps_swin"
MAX_K_PER_GPU = 3        # pool slice frees the full gallery: a sharded 4096 swin worker fits ~3/card
os.makedirs(OUT_DIR, exist_ok=True)
print(f"PS-swin: {ARM}@{CAP} {EPOCHS}ep lambda={DC_LAMBDA} avg={PS_AVG}")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Smoke-measure -> K, then launch master + K swin workers.
import json, os, subprocess, tempfile, threading, time
from pathlib import Path

logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
gpus = J.detect_gpus()
PSW = os.path.join(REPO, "Pod", "w9_ps_worker.py")
sfx = (f"psdc{int(round(DC_LAMBDA*10))}" + (f"a{PS_AVG}" if PS_AVG > 1 else "")
       + ((f"shc{int(round(COVER*100))}" if COVER > 0 else f"sh{RESHUFFLE}")
          if SHARD else "")
       + ("b" if BARRIER else "") + ("e" if EPOCH_PUSH else ""))
name = f"w9_{ARM}_g{CAP}_{sfx}_fp"

K = NWORKERS   # v3: pool slice makes workers small; no smoke gate
print(f"K={K} sharded swin workers (cover mode)", flush=True)

Path(PS_DIR).mkdir(parents=True, exist_ok=True)
def launch(role, wid, gpu):
    lg = open(logd / f"{name}_{role}{wid}.log", "w")
    cmd = ["python", "-u", PSW, "--data-dir", DATA_DIR, "--out-dir", OUT_DIR,
           "--repo", REPO, "--arm", ARM, "--anchor-cap", str(CAP),
           "--epochs", str(EPOCHS), "--ckpt-every", "50",
           "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--ps-role", role, "--ps-dir", PS_DIR, "--ps-id", str(wid),
           "--dc-lambda", str(DC_LAMBDA), "--ps-avg", str(PS_AVG)]
    if SHARD:
        cmd += ["--ps-shard", "--ps-nworkers", str(K),
                "--ps-reshuffle", str(RESHUFFLE)]
        if COVER > 0:
            cmd += ["--ps-cover", str(COVER)]
    if BARRIER:
        cmd += ["--ps-barrier"]
    if EPOCH_PUSH:
        cmd += ["--ps-epoch-push"]
    return subprocess.Popen(cmd, stdout=lg, stderr=subprocess.STDOUT,
                            env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpu))

procs = [launch("master", 0, gpus[0])]
# SEQUENCED BOOT: the master builds the 4096 gallery pack once and saves
# it; workers then LOAD it (CPU path -- no rebuild, no 17G GPU transient).
_gp = Path(DATA_DIR) / f"wscan_gal_rev_g{CAP}.npz"
t0 = time.time()
while not _gp.exists():
    assert procs[0].poll() is None, "master died during prep"
    assert time.time() - t0 < 2400, "gallery pack not saved in 40min"
    time.sleep(10)
print(f"gallery pack ready after {time.time()-t0:.0f}s", flush=True)
WGPU = ([gpus[1]] * 3 + [gpus[0]] * 2) if len(gpus) > 1 else [gpus[0]] * K
for w in range(K):
    procs.append(launch("worker", w, WGPU[w % len(WGPU)]))
    time.sleep(30)
print(f"master + {K} workers up; waiting ...", flush=True)
rc = procs[0].wait()
Path(PS_DIR, "STOP").write_text("done")
for pr in procs[1:]:
    pr.wait()
print(f"master rc={rc}; all down", flush=True)


In [ ]:
# Readout: async swin vs sync swin scratch.
import json
from pathlib import Path
sfx = (f"psdc{int(round(DC_LAMBDA*10))}" + (f"a{PS_AVG}" if PS_AVG > 1 else "")
       + ((f"shc{int(round(COVER*100))}" if COVER > 0 else f"sh{RESHUFFLE}")
          if SHARD else "")
       + ("b" if BARRIER else "") + ("e" if EPOCH_PUSH else ""))
for nm, lab in ((f"w9_{ARM}_g{CAP}_{sfx}", "ASYNC swin (PS)"),
                (f"w9_{ARM}_g{CAP}", "sync swin scratch (ref)"),
                ("w9_wcle_i2ce_icetf_g4096", "i2ce full (ref)")):
    p = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    if p.exists():
        d = json.loads(p.read_text())
        print(f"{lab:26s} ep{d['best_ep']:>4} neu {d['nm_neutral']:.3f} "
              f"non {d['nm_noname']:.3f} tag {d['tag_neutral']:.3f}/{d['tag_noname']:.3f}")
    else:
        print(f"{lab:26s} (pending)")
sp = Path(OUT_DIR) / f"ps_staleness_w9_{ARM}_g{CAP}_{sfx}_fp.json"
if sp.exists():
    h = json.loads(sp.read_text())
    drop = h.pop("dropped", 0)
    tot = sum(h.values())
    mean = sum(int(k)*c for k, c in h.items())/max(tot,1)
    print(f"staleness: mean {mean:.1f} applied {tot} dropped {drop}")


In [ ]:
# AUTO-STOP the pod (results are on the network volume).
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- stop the pod yourself.")
